# Final Aggregated Score Audit

Ce notebook reprend le script `final_aggregated_score_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Audit des scores finaux par fenetre qui peuvent alimenter une politique live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Audit final trajectory+attention+PPE aggregated risk scores.
- Commande de reproduction referencee : final aggregated score.
- Artefacts controles : Final trajectory+attention+PPE aggregated score audit exists. (`runs/exp_039_final_aggregated_score/metrics/final_score_operating_summary.csv`).
- Run par defaut : `runs/exp_039_final_aggregated_score`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "final_aggregated_score_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from ml_pipeline import ROOT, safe_auc, threshold_sweep, write_json
from sequence_experiments import append_report, make_run_dir


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `load_seed_frames`

Cette cellule definit `load_seed_frames`. Elle prepare une partie du script.

In [ ]:
def load_seed_frames(fusion_run, seed):
    files = sorted((fusion_run / "features").glob(f"same_split_fused_seed{seed}_*.csv"))
    frames = {}
    for path in files:
        match = re.match(rf"same_split_fused_seed{seed}_(.+)", path.stem)
        if match:
            frames[match.group(1)] = pd.read_csv(path)
    return frames


## Fonction `build_aggregated_frame`

Cette cellule definit `build_aggregated_frame`. Elle prepare une partie du script.

In [ ]:
def build_aggregated_frame(frames):
    if not frames:
        return pd.DataFrame()
    first_key = sorted(frames)[0]
    base_cols = [
        "row_idx",
        "video_id",
        "split",
        "frame",
        "time_s",
        "is_danger_clip",
        "target_time_s",
        "time_to_target_s",
        "danger_within_0.5s",
        "danger_within_1.0s",
        "danger_within_1.5s",
        "danger_within_2.0s",
    ]
    df = frames[first_key][base_cols + ["attention_risk", "ppe_risk"]].copy()
    seq_cols = []
    meta_cols = []
    for name, frame in sorted(frames.items()):
        df[f"sequence_{name}"] = frame["sequence_only"].to_numpy(dtype=np.float32)
        seq_cols.append(f"sequence_{name}")
        if "sequence_learned_meta_fused" in frame:
            df[f"learned_meta_{name}"] = frame["sequence_learned_meta_fused"].to_numpy(dtype=np.float32)
            meta_cols.append(f"learned_meta_{name}")
    df["sequence_mean_tcn"] = df[seq_cols].mean(axis=1)
    df["sequence_max_tcn"] = df[seq_cols].max(axis=1)
    if meta_cols:
        df["learned_meta_mean"] = df[meta_cols].mean(axis=1)

    d = df["sequence_mean_tcn"].clip(0, 1)
    a = df["attention_risk"].clip(0, 1)
    p = df["ppe_risk"].clip(0, 1)
    df["final_sequence_only"] = d
    df["final_attention_rule"] = 1.0 - (1.0 - d) * (1.0 - 0.30 * a)
    df["final_ppe_rule"] = 1.0 - (1.0 - d) * (1.0 - 0.25 * p)
    df["final_full_rule"] = 1.0 - (1.0 - d) * (1.0 - 0.30 * a) * (1.0 - 0.25 * p)
    df["final_safety_sensitive_rule"] = 1.0 - (1.0 - d) * (1.0 - 0.45 * a) * (1.0 - 0.35 * p)
    df["final_high_sensitivity_rule"] = np.clip(d + (1.0 - d) * (0.40 * a + 0.30 * p + 0.20 * a * p), 0.0, 1.0)
    df["final_attention_ppe_prior"] = np.maximum(d, np.clip(0.25 * a + 0.20 * p + 0.10 * a * p, 0.0, 1.0))
    df["final_ppe_guardrail_policy"] = np.maximum(d, np.clip(0.25 * a + 0.50 * p, 0.0, 1.0))
    if "learned_meta_mean" in df:
        df["final_learned_meta_mean"] = df["learned_meta_mean"]
    return df


## Fonction `evaluate_score`

Cette cellule definit `evaluate_score`. Elle prepare une partie du script.

In [ ]:
def evaluate_score(df, score_col, seed, persistence_windows):
    rows = []
    sweeps = []
    for split_name in ["train", "val", "test"]:
        split_df = df[df["split"] == split_name].copy()
        y = split_df["danger_within_1.0s"].astype(int).to_numpy()
        p = split_df[score_col].to_numpy()
        sweep = threshold_sweep(split_df.rename(columns={score_col: "risk"}), "risk", 1.0, split_name, persistence_windows=persistence_windows)
        sweep["repeat_seed"] = seed
        sweep["score_variant"] = score_col
        sweeps.append(sweep)
        best = sweep.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
        rows.append(
            {
                "repeat_seed": seed,
                "score_variant": score_col,
                "split": split_name,
                "average_precision": safe_auc(average_precision_score, y, p),
                "roc_auc": safe_auc(roc_auc_score, y, p),
                "best_threshold_by_hit_fa": float(best["threshold"]),
                "best_hit_rate": float(best["danger_clip_hit_rate"]),
                "best_false_alarms_per_min": float(best["safe_false_alarms_per_min"]),
                "best_window_precision": float(best["window_precision"]),
                "best_window_recall": float(best["window_recall"]),
                "best_median_early_warning_s": best["median_early_warning_s"],
            }
        )
    return rows, sweeps


## Fonction `select_validation_threshold`

Cette cellule definit `select_validation_threshold`. Elle prepare une partie du script.

In [ ]:
def select_validation_threshold(sweep, policy):
    val = sweep[sweep["split"] == "val"].copy()
    if policy == "max_hit_low_fa":
        return val.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
    if policy == "balanced_f1":
        return val.sort_values(["window_f1", "danger_clip_hit_rate", "safe_false_alarms_per_min"], ascending=[False, False, True]).iloc[0]
    if policy == "precision_guard":
        eligible = val[val["window_precision"] >= 0.50]
        if eligible.empty:
            eligible = val
        return eligible.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
    raise ValueError(policy)


## Fonction `apply_threshold`

Cette cellule definit `apply_threshold`. Elle prepare une partie du script.

In [ ]:
def apply_threshold(df, score_col, threshold, seed, policy, persistence_windows):
    test = df[df["split"] == "test"].copy()
    sweep = threshold_sweep(test.rename(columns={score_col: "risk"}), "risk", 1.0, "test", persistence_windows=persistence_windows)
    exact = sweep.iloc[(sweep["threshold"] - threshold).abs().argsort().iloc[0]]
    y = test["danger_within_1.0s"].astype(int).to_numpy()
    p = test[score_col].to_numpy()
    return {
        "repeat_seed": seed,
        "score_variant": score_col,
        "policy": policy,
        "selected_threshold": float(threshold),
        "test_average_precision": safe_auc(average_precision_score, y, p),
        "test_roc_auc": safe_auc(roc_auc_score, y, p),
        "test_hit_rate": float(exact["danger_clip_hit_rate"]),
        "test_false_alarms_per_min": float(exact["safe_false_alarms_per_min"]),
        "test_window_precision": float(exact["window_precision"]),
        "test_window_recall": float(exact["window_recall"]),
        "test_window_f1": float(exact["window_f1"]),
        "test_median_early_warning_s": exact["median_early_warning_s"],
    }


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(df, group_cols, metric_cols):
    rows = []
    for keys, group in df.groupby(group_cols):
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        for col in metric_cols:
            row[f"{col}_mean"] = float(pd.to_numeric(group[col], errors="coerce").mean())
            row[f"{col}_std"] = float(pd.to_numeric(group[col], errors="coerce").std(ddof=0))
        rows.append(row)
    return pd.DataFrame(rows)


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    fusion_run = resolve(args.fusion_run)
    run_dir = make_run_dir(args.run_name)
    seeds = []
    for path in (fusion_run / "features").glob("same_split_fused_seed*_*.csv"):
        match = re.search(r"seed(\d+)_", path.name)
        if match:
            seeds.append(int(match.group(1)))
    seeds = sorted(set(seeds))
    write_json(
        run_dir / "config.json",
        {
            "fusion_run": str(fusion_run),
            "seeds": seeds,
            "persistence_windows": args.persistence_windows,
            "score_definitions": {
                "final_sequence_only": "mean of same-split TCN sequence risks",
                "final_full_rule": "1-(1-danger)*(1-0.30*attention)*(1-0.25*ppe)",
                "final_safety_sensitive_rule": "1-(1-danger)*(1-0.45*attention)*(1-0.35*ppe)",
                "final_high_sensitivity_rule": "danger plus stronger residual attention/PPE boosts",
                "final_attention_ppe_prior": "max(sequence risk, attention/PPE prior)",
                "final_ppe_guardrail_policy": "max(sequence risk, 0.25*attention + 0.50*ppe)",
            },
        },
    )
    score_cols = [
        "final_sequence_only",
        "final_attention_rule",
        "final_ppe_rule",
        "final_full_rule",
        "final_safety_sensitive_rule",
        "final_high_sensitivity_rule",
        "final_attention_ppe_prior",
        "final_ppe_guardrail_policy",
        "final_learned_meta_mean",
    ]
    all_metrics = []
    all_sweeps = []
    selected_rows = []
    for seed in seeds:
        frames = load_seed_frames(fusion_run, seed)
        df = build_aggregated_frame(frames)
        if df.empty:
            continue
        df.to_csv(run_dir / "features" / f"final_scores_seed{seed}.csv", index=False)
        available_scores = [col for col in score_cols if col in df]
        seed_sweeps = {}
        for score_col in available_scores:
            rows, sweeps = evaluate_score(df, score_col, seed, args.persistence_windows)
            all_metrics.extend(rows)
            sweep_df = pd.concat(sweeps, ignore_index=True)
            seed_sweeps[score_col] = sweep_df
            all_sweeps.append(sweep_df)
            for policy in ["max_hit_low_fa", "balanced_f1", "precision_guard"]:
                selected = select_validation_threshold(sweep_df, policy)
                selected_rows.append(apply_threshold(df, score_col, float(selected["threshold"]), seed, policy, args.persistence_windows))
    metrics = pd.DataFrame(all_metrics)
    sweeps = pd.concat(all_sweeps, ignore_index=True) if all_sweeps else pd.DataFrame()
    selected = pd.DataFrame(selected_rows)
    metrics.to_csv(run_dir / "metrics" / "final_score_metrics.csv", index=False)
    sweeps.to_csv(run_dir / "metrics" / "final_score_threshold_sweeps.csv", index=False)
    selected.to_csv(run_dir / "metrics" / "final_score_validation_selected_thresholds.csv", index=False)
    metric_summary = summarize(
        metrics,
        ["score_variant", "split"],
        ["average_precision", "roc_auc", "best_hit_rate", "best_false_alarms_per_min", "best_window_precision"],
    )
    selected_summary = summarize(
        selected,
        ["score_variant", "policy"],
        ["selected_threshold", "test_average_precision", "test_hit_rate", "test_false_alarms_per_min", "test_window_precision", "test_window_f1", "test_median_early_warning_s"],
    )
    metric_summary.to_csv(run_dir / "metrics" / "final_score_metric_summary.csv", index=False)
    selected_summary.to_csv(run_dir / "metrics" / "final_score_operating_summary.csv", index=False)

    lines = ["# Final Aggregated Score Audit", ""]
    lines.append("This audit defines final operational scores that combine trajectory danger, attention risk, and PPE/blouse risk on shared parent-video splits.")
    lines.append("")
    lines.append("## Validation-Selected Test Operating Points")
    lines.append("")
    lines.append("| score | policy | threshold | test AP | hit | FA/min | precision | median early s |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|")
    view = selected_summary.copy()
    view["rank"] = (
        view["test_average_precision_mean"]
        + 0.8 * view["test_hit_rate_mean"]
        + 0.3 * view["test_window_precision_mean"]
        - 0.03 * view["test_false_alarms_per_min_mean"]
    )
    for _, row in view.sort_values("rank", ascending=False).head(21).iterrows():
        lines.append(
            f"| {row['score_variant']} | {row['policy']} | {row['selected_threshold_mean']:.2f} | "
            f"{row['test_average_precision_mean']:.3f} | {row['test_hit_rate_mean']:.3f} | "
            f"{row['test_false_alarms_per_min_mean']:.3f} | {row['test_window_precision_mean']:.3f} | "
            f"{row['test_median_early_warning_s_mean']:.3f} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- Attention/PPE modifiers can increase sensitivity, but the stronger modifiers usually trade precision for false alarms.")
    lines.append("- The final operational score should be a named policy choice, not a hidden threshold.")
    lines.append("- The sequence-only ensemble remains the cleanest danger model; fused rule scores are useful when the product should deliberately become more sensitive for distracted or poorly worn PPE states.")
    lines.append("")
    lines.append("## Artifacts")
    lines.append("")
    lines.append(f"- Per-seed final scores: `{run_dir / 'features'}`")
    lines.append(f"- Metrics: `{run_dir / 'metrics' / 'final_score_metrics.csv'}`")
    lines.append(f"- Validation-selected thresholds: `{run_dir / 'metrics' / 'final_score_validation_selected_thresholds.csv'}`")
    summary_path = run_dir / "final_aggregated_score_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Final Aggregated Score Audit", f"- Summary: `{summary_path}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Audit final trajectory+attention+PPE aggregated risk scores.")
    parser.add_argument("--fusion-run", default="runs/exp_020_same_split_fusion")
    parser.add_argument("--run-name", default="exp_039_final_aggregated_score")
    parser.add_argument("--persistence-windows", type=int, default=2)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_039_final_aggregated_score_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["final_aggregated_score_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
